In [ ]:
import os
import torch
from torch.distributions import Normal

import numpy as np
from tv_snr.snr_schedules import *
from tv_snr.scale_schedules import *
from tv_snr.samplers import SNREulerParamScore
from tv_snr.snr import Scale_SNR_SDE
from tv_snr.time_schedules import KVeSchedule
from tv_snr.constants import image_key
import matplotlib.pyplot as plt
from tv_snr.noise_schedules import *

os.makedirs("figures", exist_ok=True)

eps = 2e-4

def test_case(testcase = "OTFM", T=16):
    sigma_max = 1.0
    time_sched = KVeSchedule(sigma_max=1.0, sigma_min=0.0, discretize=True, T=T)
    if testcase == "OTFM":
        snr_sch = InverseSigmoid(slope=2, shift=0., t_min=0.001, t_max=0.999)
        scale_sch = FMScale(snr_sch)
        sde = Scale_SNR_SDE(
            snr_sch=snr_sch,
            scale_sch=scale_sch,
            invariant=False,
            disc_type="forward",
            log_deriv=True
        )
    elif testcase == "VP-OTFM":
        snr_sch = InverseSigmoid(slope=2, shift=0., t_min=0.001, t_max=0.999)
        scale_sch = ConstScale(snr_sch)
        sde = Scale_SNR_SDE(
            snr_sch=snr_sch,
            scale_sch=scale_sch,
            invariant=False,
            disc_type="forward",
            log_deriv=True
        )
    elif testcase == "EDM":
        snr_sch = LinearToSNRSchedule(sigma_min=eps, sigma_max=80.0)
        sde = Scale_SNR_SDE(
            snr_sch=snr_sch,
            scale_sch=VeScale(snr_sch),
            invariant=False,
            disc_type="forward",
            log_deriv=True
        )
        sigma_max = 80.0
        time_sched = KVeSchedule(sigma_max=1.0, sigma_min=0.0, discretize=True, T=T)
    elif testcase == "EDM-UT":
        snr_sch = KveToSNRSchedule(t_min=0.0, t_max=1.0, sigma_max=80.0, sigma_min=eps)
        sde = Scale_SNR_SDE(
            snr_sch=snr_sch,
            scale_sch=VeScale(snr_sch),
            invariant=False,
            disc_type="forward",
            log_deriv=True
        )
        time_sched = None
        sigma_max = 80.0
    elif testcase == "VP-EDM":
        snr_sch = LinearToSNRSchedule(sigma_min=eps, sigma_max=80.0)
        sde = Scale_SNR_SDE(
            snr_sch=snr_sch,
            scale_sch=ConstScale(snr_sch),
            invariant=False,
            disc_type="forward",
            log_deriv=True
        )
    elif testcase == "VP-EDM-UT":
        snr_sch = KveToSNRSchedule(t_min=0.0, t_max=1.0, sigma_max=80.0, sigma_min=eps)
        sde = Scale_SNR_SDE(
            snr_sch=snr_sch,
            scale_sch=ConstScale(snr_sch),
            invariant=False,
            disc_type="forward",
            log_deriv=True
        )
    elif testcase == "VP-ISSNR[$\eta$=1 $\kappa$=2]":
        snr_sch = InverseSigmoid(slope=2, shift=4., t_min=0.001, t_max=0.999)
        scale_sch = ConstScale(snr_sch)
        sde = Scale_SNR_SDE(
            snr_sch=snr_sch,
            scale_sch=scale_sch,
            invariant=False,
            disc_type="forward",
            log_deriv=True
        )
    elif testcase == "VP-ISSNR[$\eta$=1.5 $\kappa$=1]":
        snr_sch = InverseSigmoid(slope=3, shift=2., t_min=0.001, t_max=0.999)
        scale_sch = ConstScale(snr_sch)
        sde = Scale_SNR_SDE(
            snr_sch=snr_sch,
            scale_sch=scale_sch,
            invariant=False,
            disc_type="forward",
            log_deriv=True
        )
    elif testcase == "SMLD":
        snr_sch = VeToSNRSchedule(t_min=0, t_max=1.0, sigma_max=80.0, sigma_min=eps)
        sde = Scale_SNR_SDE(
            snr_sch=snr_sch,
            scale_sch=VeScale(snr_sch),
            invariant=False,
            disc_type="forward",
            log_deriv=True
        )
        time_sched = None
        sigma_max = 80.0
    elif testcase == "VP-SMLD":
        snr_sch = VeToSNRSchedule(t_min=0, t_max=1.0, sigma_max=80.0, sigma_min=eps)
        sde = Scale_SNR_SDE(
            snr_sch=snr_sch,
            scale_sch=ConstScale(snr_sch),
            invariant=False,
            disc_type="forward",
            log_deriv=True
        )
    elif testcase == "DDPM cos-v2.5":
        noise_sch = CosineSchedule(v=2.5, s=1e-5, T=T, discretize=True)
        snr_sch = NoiseToSNRSchedule(noise_sch, t_min=0.001, t_max=0.999)
        sde = Scale_SNR_SDE(
            snr_sch=snr_sch,
            scale_sch=ConstScale(snr_sch),
            invariant=False,
            disc_type="forward",
            log_deriv=True
        )
    elif testcase == "DDPM cos-v1.0":
        noise_sch = CosineSchedule(v=1.0, s=1e-5, T=T, discretize=True)
        snr_sch = NoiseToSNRSchedule(noise_sch, t_min=0.001, t_max=0.999)
        sde = Scale_SNR_SDE(
            snr_sch=snr_sch,
            scale_sch=ConstScale(snr_sch),
            invariant=False,
            disc_type="forward",
            log_deriv=True
        )
    elif testcase == "DDPM linear":
        linear = LinearSchedule(beta_start=0.001, beta_end=20.0, discretize=True, T=T)
        snr_linear = NoiseToSNRSchedule(linear,t_min=0.0001,t_max=0.999)
        sde = Scale_SNR_SDE(
            snr_sch=snr_linear,
            scale_sch=ConstScale(snr_linear),
            invariant=False,
            disc_type="forward",
            log_deriv=True
        )
    else:
        raise ValueError(f"Unknown testcase {testcase}")

    return sde, sde.reverse(stochastic=False), sigma_max, time_sched

In [ ]:
def second_derivative(x: torch.Tensor, f: torch.Tensor):
    """
    Compute the second derivative of f with respect to x using finite differences.
    
    Parameters:
    x: torch.Tensor - 1D tensor of x-values (assumed sorted, but not necessarily uniform)
    f: torch.Tensor - 1D tensor of function values f(x)
    
    Returns:
    d2f_dx2: torch.Tensor - 1D tensor of second derivative values
    """

    h_plus = x[2:] - x[1:-1]   # x_{i+1} - x_i
    h_minus = x[1:-1] - x[:-2] # x_i - x_{i-1}

    f_plus = f[2:]   # f(x_{i+1})
    f_mid = f[1:-1]  # f(x_i)
    f_minus = f[:-2] # f(x_{i-1})

    # Compute second derivative
    d2f_dx2 = 2 / (h_plus + h_minus) * ((f_plus - f_mid) / h_plus - (f_mid - f_minus) / h_minus)

    # Pad with NaN or approximate boundaries
    d2f_dx2 = torch.cat([torch.tensor([float('nan')]), d2f_dx2, torch.tensor([float('nan')])])

    return d2f_dx2

def first_derivative(x: torch.Tensor, f: torch.Tensor):
    """
    Compute the first derivative of f with respect to x using finite differences.
    
    Parameters:
    x: torch.Tensor - 1D tensor of x-values (assumed sorted, but not necessarily uniform)
    f: torch.Tensor - 1D tensor of function values f(x)
    
    Returns:
    df_dx: torch.Tensor - 1D tensor of first derivative values
    """
        
    df_dx = torch.zeros_like(f)

    # Central difference for interior points
    df_dx[1:-1] = (f[2:] - f[:-2]) / (x[2:] - x[:-2])

    # One-sided differences for boundaries
    df_dx[0] = (f[1] - f[0]) / (x[1] - x[0])  # Forward difference at the start
    df_dx[-1] = (f[-1] - f[-2]) / (x[-1] - x[-2])  # Backward difference at the end

    return df_dx

In [ ]:
from itertools import product
from matplotlib.colors import LogNorm
import matplotlib.colors as mcolors
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from torch.distributions import MixtureSameFamily, Categorical

T = 92
vmin = -12
contours = True

#ccs = [["edm", "vp-issnr[eta=1,k=2]", "vp-issnr[eta=1.5,k=1]", "cos-v2.5", "cos-v1.0"], ["smld", "vp-smld", "otfm", "vp-otfm", "edm-ut", "vp-edm-ut"]]
# appendix: all schedules
ccs = [["SMLD", "VP-SMLD", "OTFM", "VP-OTFM", "EDM-UT", "VP-EDM-UT"], ["EDM", "DDPM cos-v2.5", "DDPM cos-v1.0", "DDPM linear", "VP-ISSNR[$\eta$=1 $\kappa$=2]", "VP-ISSNR[$\eta$=1.5 $\kappa$=1]"]]
#ccs = [["OTFM"]]
n_data_points = [3]

for iiii, (ccc, ndp) in enumerate(product(ccs, n_data_points)):
    # individual plots follow

    f, axs = plt.subplots(1, len(ccc), figsize=(3 * len(ccc), 3), squeeze=False)
    axs = axs.flatten()

    for testc, ax in zip(ccc, axs):
        sde, rsde, sigma_max, time_sched = test_case(testc, T)
        nnn = ndp * 3 if ndp > 5 else 24  

        prior = Normal(torch.tensor(0), torch.ones(1,))
        samples = prior.icdf(torch.linspace(0.0, 1.0, nnn + 2)[1:-1])
        samples = samples.reshape(nnn, 1, 1, 1)

        class AnalyticScoreMultiplePoints(torch.nn.Module):
            def __init__(self, n_points):
                self.n_points = n_points
                self.points_x0 = torch.linspace(-1, 1, self.n_points).float().reshape(-1, 1)

                # standardize the points
                std = torch.sqrt(torch.square(self.points_x0).sum()/self.points_x0.shape[0])
                self.points_x0 = self.points_x0 / std

                super().__init__()
            
            def forward(self, x, t):
                x_shape = x.shape
                x = x.reshape(-1)
                t = t.reshape(-1)

                x = x.clone().detach().requires_grad_(True)
                mog = self.log_prob(x, t)
                score = torch.autograd.grad(mog.sum(), x, create_graph=False)[0]
                score = score.reshape(x_shape)
                return score

            def log_prob(self, x, t):
                t = t.reshape(1, -1)

                means, stds = sde.perturbation_kernel(self.points_x0.to(x.device), t)
                logprobs = Normal(means, stds).log_prob(x)
                logprobs = torch.where(logprobs < -100, torch.tensor(-100, device=x.device).float(), logprobs)

                return torch.log(torch.exp(logprobs).mean(dim=0))

        net = AnalyticScoreMultiplePoints(ndp)

        sampler = SNREulerParamScore(
            T=T,
            time_schedule=time_sched,
            reverse_process = rsde,
            denoiser = net,
            out_var_scaler=1.0,
            scale_input=False,
            snr_key="gamma",
            std_key = "sigma",
            noise_pred_key = "eps_pred",
            max_stoch_std=float('inf'),
            min_stoch_std=0.0,
            clip_stoch_std=False,
            selected_stoch=False,
            save_progress=True,
            conditional=False
        ) 

        ymaxx = 2.0 * sigma_max

        # if testc == "EDM":
        #     tt = np.logspace(-5.5, 0, 45)
        #     xx = np.linspace(-1.0, 1.0, 55) * ymaxx

        #     maxp, minp = net.points_x0.max().item(), net.points_x0.min().item()
        #     xx = np.concatenate([xx, np.linspace(minp * 2.3, maxp * 2.3, 15)])
        # elif sigma_max > 1.0:
        tt = np.linspace(0.0001, 0.99, 55)
        xx = np.linspace(-1.0, 1.0, 55) * ymaxx
        # else:
        #     tt = np.logspace(-5.5, 0, 45)
        #     xx = np.linspace(-1.0, 1.0, 55) * ymaxx

        xx = np.concatenate([xx, net.points_x0.numpy().squeeze() * 0.996])
        xx = np.concatenate([xx, net.points_x0.numpy().squeeze() * 1.004])
        xx = np.sort(np.concatenate([xx, net.points_x0.numpy().squeeze()]))
        TT, XX = np.meshgrid(tt, xx)

        Z = net.log_prob(torch.from_numpy(XX.reshape(-1)), torch.from_numpy(TT.reshape(-1)))
        Z = Z.reshape((TT.shape[0], TT.shape[1])).numpy()

        oranges = plt.cm.Oranges(np.linspace(0, 1, 256))
        oranges[0] = [1, 1, 1, 1]
        custom_oranges = mcolors.LinearSegmentedColormap.from_list("CustomOranges", oranges)

        if contours:
            ax.contourf(TT, XX, Z, levels=500, vmin=vmin, cmap=custom_oranges)
        levels = np.linspace(vmin, np.max(Z), 10)
        #ax.contour(TT, XX, Z, levels=levels, colors='#FFCCCC', linewidths=0.5)

        result = sampler.denoise({image_key: samples * sigma_max})
        traj = result[2]["image"]

        toplotx = torch.linspace(1, 0, traj[0].shape[-1])
        if time_sched is not None:
            toplotx = time_sched(toplotx).numpy()

        for i in range(traj.shape[0]):
            toplot = traj[i].squeeze().detach().numpy()
            #toplotx = torch.linspace(1, 0, toplot.shape[0])
            
            if i == traj.shape[0] - 1:
                pass
                # uncomment to plot marker points
                #ax.plot(toplotx, toplot, c="black", linewidth=0.5, marker='o', markersize=0.8)
            else:
                ax.plot(toplotx, toplot, c="black", linewidth=0.5)

        # PLOT 2nd derivative
        #inset_ax_t2x = inset_axes(ax, width="100%", height="10%", loc="lower center", bbox_to_anchor=(0.0, -0.35, 1.0, 1.0), bbox_transform=ax.transAxes)
        toplotx = torch.tensor(toplotx)
        first_index = (toplotx < 1e-2).nonzero(as_tuple=True)[0][0]
        toplotxx = toplotx[:first_index]

        all_twodx = []
        for i in range(traj.shape[0]):
            curve = traj[i].squeeze()[:first_index]
            twodx = second_derivative(toplotxx, curve)
            all_twodx.append(twodx)
        twodx_agg = torch.stack(all_twodx).abs().mean(dim=0)
        #inset_ax_t2x.plot(toplotxx, twodx_agg, c="blue", linewidth=0.9)
        #inset_ax_t2x.axis('off')  # Disable axes
        #inset_ax_t2x.title.set_text("Curvature")
        #inset_ax_t2x.set_ylim(0, 5.0)
        #inset_ax_t2x.title.set_fontsize(11)

        #inset_ax_t2x.set_facecolor((1, 1, 1, 0))  # Make background transparent
        #inset_ax_t2x.set_position(ax.get_position())  # Align exactly with parent ax
        # -------------------

        ## GET rectified flow measure integral
        all_measure = []
        for i in range(traj.shape[0]):
            curve = traj[i].squeeze()[:first_index]
            dx = first_derivative(toplotxx, curve)
            all_measure.append(torch.square((curve[0] - curve[-1]) - dx))
        dx_agg = torch.stack(all_measure).mean(dim=0)
        integral = torch.trapz(reversed(dx_agg), reversed(toplotxx))
        
        inset_ax_rflow = inset_axes(ax, width="100%", height="32%", loc="lower center", bbox_to_anchor=(0.0, -0.6, 1.0, 1.0), bbox_transform=ax.transAxes)
        inset_ax_rflow.plot(toplotxx, dx_agg, c="green", linewidth=1.8)
        inset_ax_rflow.title.set_text(f"Curvature (integral: {integral.item():.2f})")
        inset_ax_rflow.title.set_fontsize(10)
        inset_ax_rflow.set_ylim(-2, 12)
        inset_ax_rflow.xaxis.set_visible(False)
        inset_ax_rflow.axhline(0, color='black', linestyle='--', linewidth=0.8)
        #inset_ax_support.xaxis.set_visible(False)
        ###

        ## PLOT support
        inset_ax_support = inset_axes(ax, width="100%", height="32%", loc="lower center", bbox_to_anchor=(0.0, -1.1, 1.0, 1.0), bbox_transform=ax.transAxes)
        base_support = sampler.reverse_process.forward_process.get_b(torch.tensor(1.0))
        support = sampler.reverse_process.forward_process.get_b(toplotxx)
        support = support / base_support
        inset_ax_support.plot(toplotxx, support, c="red", linewidth=1.8)
        # inset_ax_support.set_facecolor((1, 1, 1, 0))  # Make background transparent
        # inset_ax_support.set_position(ax.get_position())  # Align exactly with parent ax
        inset_ax_support.title.set_text("Support")
        inset_ax_support.set_ylim(-0.1, 1.1)
        inset_ax_support.title.set_fontsize(11)
        ## PLOT support END
        

        
        ticks = np.linspace(0, 1, traj.shape[0])

        if testc == "EDM":
            ticks = kve_schedule(ticks, sigma_min=0.0, sigma_max=1.0)
            # ax.set_xticks(ticks)
            # ax.set_xticklabels([(f"{x:.2f}" if (i == 0 or i == 20 or i == 18) else "") for (i, x) in enumerate(ticks)])

        # else:
        #     ax.set_xticks(ticks)
        #     ax.set_xticklabels([(f"{x:.2f}" if i % 10 == 0 else "") for (i, x) in enumerate(ticks)])
        ax.set_xticks(ticks)
        ax.set_xticklabels([(f"{x:.2f}" if (x == 0 or x == 1.0) else "") for (i, x) in enumerate(ticks)])

        #if sigma_max > 1.0:
        # plot again
        inset_ax = inset_axes(ax, width="37%", height="37%", loc="upper left", bbox_to_anchor=(0.12, -0.11, 1.0, 1.0), bbox_transform=ax.transAxes)

        inset_ax.yaxis.tick_left()
        inset_ax.xaxis.tick_top()

        topx0 = net.points_x0[-1].item()
        # if testc == "EDM" or testc == "SMLD" or testc == "EDM-UT":
        #     inset_ax.set_ylim(-3, 3)
        #     inset_ax.set_yticks([-2, 0, 2])
        #     ymin = -3
        #     ymax = 3
        # el
        # if testc == "VP-ISSNR[$\eta$=1 $\kappa$=2]" or testc == "VP-ISSNR[$\eta$=1.5 $\kappa$=1]" or testc == "VP-SMLD" or testc == "VP-EDM-UT":
        #     delta = 0.01
        #     inset_ax.set_ylim(topx0 - delta, topx0 + delta)
        #     inset_ax.set_yticks([np.round(topx0 - delta * 0.5, decimals=2), np.round(topx0 + delta * 0.5, decimals=2)])
        #     ymin = topx0 - delta
        #     ymax = topx0 + delta
        # else:
        deccc = 2

        if testc == "SMLD":
            delta = 0.01 # 0.03
            deccc = 3
        elif testc == "VP-SMLD":
            delta = 0.01 # 0.03
            deccc = 3
        elif testc == "OTFM":
            delta = 0.08
        elif testc == "VP-OTFM":
            delta = 0.08
        elif testc == "EDM-UT":
            delta = 0.02
        elif testc == "VP-EDM-UT":
            delta = 0.02
        elif testc == "EDM":
            delta = 1.05
        elif testc == "DDPM cos-v2.5":
            delta = 0.002
            deccc = 3
        elif testc == "DDPM cos-v1.0":
            delta = 0.3
        elif testc == "DDPM linear":
            delta = 0.5
        elif testc == "VP-ISSNR[$\eta$=1 $\kappa$=2]":
            delta = 0.03
        elif testc == "VP-ISSNR[$\eta$=1.5 $\kappa$=1]":
            delta = 0.015
        else:
            raise ValueError(f"Unknown testcase {testc}")

        # if testc == "EDM":
        #     delta = 1.05
        # elif testc == "DDPM cos-v1.0" or testc == "DDPM linear":
        #     delta = 0.5
        # elif "OTFM" in testc:
        #     delta = 0.3
        # elif testc == "DDPM cos-v2.5":
        #     delta = 0.003
        # else: #  or testc.startswith("VP-ISSNR") or testc.index("SMLD") > -1
        #     delta = 0.03

        inset_ax.set_ylim(topx0 - delta, topx0 + delta)
        paddd = -32 if (deccc == 3) else -25
        inset_ax.set_yticks([np.round(topx0 - delta * 0.5, decimals=deccc), np.round(topx0 + delta * 0.5, decimals=deccc)])
        ymin = topx0 - delta
        ymax = topx0 + delta

        inset_ax.yaxis.set_tick_params(direction='in', pad=paddd) 
        inset_ax.set_xticks(ticks) # [0.0, 0.04]
        inset_ax.set_xticklabels([(f"{x:.2f}" if (x == 0.0 or x >= 0.03) else "") for (i, x) in enumerate(ticks)])

        inset_ax.set_xlim(-0.0065, 0.05)

        tt = np.logspace(-4, np.log10(0.05), 45)
        xx = np.linspace(ymin, ymax, 55)
        TT, XX = np.meshgrid(tt, xx)

        Z = net.log_prob(torch.from_numpy(XX.reshape(-1)), torch.from_numpy(TT.reshape(-1)))
        Z = Z.reshape((TT.shape[0], TT.shape[1])).numpy()

        if contours:
            inset_ax.contourf(TT, XX, Z, levels=1000, vmin=vmin / 2, cmap=custom_oranges)

        for i in range(traj.shape[0]):
            toplot = traj[i].squeeze().detach().numpy()
            #toplotx = torch.linspace(1, 0, toplot.shape[0])

            if i == traj.shape[0] - 1:
                pass
                # uncomment to plot marker points
                # inset_ax.plot(toplotx, toplot, c="black", linewidth=0.3, marker='o', markersize=2)
            else:
                inset_ax.plot(toplotx, toplot, c="black", linewidth=0.3)

        # end inset

        ax.set_ylim(-ymaxx, ymaxx)
        title = f"{testc}"
        if testc[0] == "V":
            title = "Our " + title
        ax.set_title(title)  # - {ndp} deltas
    
    plt.savefig(f"figures/toy_example{iiii}_{ndp}_deltas.pdf", bbox_inches='tight', pad_inches=0)
    plt.savefig(f"figures/toy_example{iiii}_{ndp}_deltas.png", bbox_inches='tight', pad_inches=0)
    plt.tight_layout()
    plt.show()

In [ ]:
from itertools import product
from matplotlib.colors import LogNorm
import matplotlib.colors as mcolors
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from torch.distributions import MixtureSameFamily, Categorical
from matplotlib.gridspec import GridSpec

T = 92
vmin = -12
contours = True

#ccs = [["edm", "vp-issnr[eta=1,k=2]", "vp-issnr[eta=1.5,k=1]", "cos-v2.5", "cos-v1.0"], ["smld", "vp-smld", "otfm", "vp-otfm", "edm-ut", "vp-edm-ut"]]
#ccs = [["SMLD", "VP-SMLD", "OTFM", "VP-OTFM", "EDM-UT", "VP-EDM-UT"], ["EDM", "DDPM cos-v2.5", "DDPM cos-v1.0", "DDPM linear", "VP-ISSNR[$\eta$=1 $\kappa$=2]", "VP-ISSNR[$\eta$=1.5 $\kappa$=1]"]]
ccs = [["EDM", "OTFM", "VP-ISSNR[$\eta$=1 $\kappa$=2]"]]
#ccs = [["OTFM"]]
n_data_points = [3]

for iiii, (ccc, ndp) in enumerate(product(ccs, n_data_points)):
    # individual plots follow

    # f, axs = plt.subplots(1, len(ccc), figsize=(3 * len(ccc), 2.3), squeeze=False)
    # axs = axs.flatten()


    ncols = 1
    nrows = 2 * len(ccc)  # two rows per testc
    fig = plt.figure(figsize=(4, 3* len(ccc)))
    gs = GridSpec(nrows, ncols, height_ratios=[3, 1] * len(ccc), figure=fig, hspace=0.3)

#    for testc, ax in zip(ccc, axs):
    # for idx, testc in enumerate(ccc):
    #     ax = axs[2 * idx]
    #     inset_ax_combined = axs[2 * idx + 1]

    for idx, testc in enumerate(ccc):
        ax = fig.add_subplot(gs[2 * idx, 0])
        inset_ax_combined = fig.add_subplot(gs[2 * idx + 1, 0])

        sde, rsde, sigma_max, time_sched = test_case(testc, T)
        nnn = ndp * 3 if ndp > 5 else 24  

        prior = Normal(torch.tensor(0), torch.ones(1,))
        samples = prior.icdf(torch.linspace(0.0, 1.0, nnn + 2)[1:-1])
        samples = samples.reshape(nnn, 1, 1, 1)

        class AnalyticScoreMultiplePoints(torch.nn.Module):
            def __init__(self, n_points):
                self.n_points = n_points
                self.points_x0 = torch.linspace(-1, 1, self.n_points).float().reshape(-1, 1)

                # standardize the points
                std = torch.sqrt(torch.square(self.points_x0).sum()/self.points_x0.shape[0])
                self.points_x0 = self.points_x0 / std

                super().__init__()
            
            def forward(self, x, t):
                x_shape = x.shape
                x = x.reshape(-1)
                t = t.reshape(-1)

                x = x.clone().detach().requires_grad_(True)
                mog = self.log_prob(x, t)
                score = torch.autograd.grad(mog.sum(), x, create_graph=False)[0]
                score = score.reshape(x_shape)
                return score

            def log_prob(self, x, t):
                t = t.reshape(1, -1)

                means, stds = sde.perturbation_kernel(self.points_x0.to(x.device), t)
                logprobs = Normal(means, stds).log_prob(x)
                logprobs = torch.where(logprobs < -100, torch.tensor(-100, device=x.device).float(), logprobs)

                return torch.log(torch.exp(logprobs).mean(dim=0))

        net = AnalyticScoreMultiplePoints(ndp)

        sampler = SNREulerParamScore(
            T=T,
            time_schedule=time_sched,
            reverse_process = rsde,
            denoiser = net,
            out_var_scaler=1.0,
            scale_input=False,
            snr_key="gamma",
            std_key = "sigma",
            noise_pred_key = "eps_pred",
            max_stoch_std=float('inf'),
            min_stoch_std=0.0,
            clip_stoch_std=False,
            selected_stoch=False,
            save_progress=True,
            conditional=False
        ) 

        ymaxx = 2.0 * sigma_max

        # if testc == "EDM":
        #     tt = np.logspace(-5.5, 0, 45)
        #     xx = np.linspace(-1.0, 1.0, 55) * ymaxx

        #     maxp, minp = net.points_x0.max().item(), net.points_x0.min().item()
        #     xx = np.concatenate([xx, np.linspace(minp * 2.3, maxp * 2.3, 15)])
        # elif sigma_max > 1.0:
        tt = np.linspace(0.0001, 0.99, 55)
        xx = np.linspace(-1.0, 1.0, 55) * ymaxx
        # else:
        #     tt = np.logspace(-5.5, 0, 45)
        #     xx = np.linspace(-1.0, 1.0, 55) * ymaxx

        xx = np.concatenate([xx, net.points_x0.numpy().squeeze() * 0.996])
        xx = np.concatenate([xx, net.points_x0.numpy().squeeze() * 1.004])
        xx = np.sort(np.concatenate([xx, net.points_x0.numpy().squeeze()]))
        TT, XX = np.meshgrid(tt, xx)

        Z = net.log_prob(torch.from_numpy(XX.reshape(-1)), torch.from_numpy(TT.reshape(-1)))
        Z = Z.reshape((TT.shape[0], TT.shape[1])).numpy()

        oranges = plt.cm.Oranges(np.linspace(0, 1, 256))
        oranges[0] = [1, 1, 1, 1]
        custom_oranges = mcolors.LinearSegmentedColormap.from_list("CustomOranges", oranges)

        if contours:
            ax.contourf(TT, XX, Z, levels=500, vmin=vmin, cmap=custom_oranges)
        levels = np.linspace(vmin, np.max(Z), 10)
        #ax.contour(TT, XX, Z, levels=levels, colors='#FFCCCC', linewidths=0.5)

        result = sampler.denoise({image_key: samples * sigma_max})
        traj = result[2]["image"]

        toplotx = torch.linspace(1, 0, traj[0].shape[-1])
        if time_sched is not None:
            toplotx = time_sched(toplotx).numpy()

        for i in range(traj.shape[0]):
            toplot = traj[i].squeeze().detach().numpy()
            #toplotx = torch.linspace(1, 0, toplot.shape[0])
            
            if i == traj.shape[0] - 1:
                pass
                # uncomment to plot marker points
                #ax.plot(toplotx, toplot, c="black", linewidth=0.5, marker='o', markersize=0.8)
            else:
                ax.plot(toplotx, toplot, c="black", linewidth=0.5)

        # PLOT 2nd derivative
        #inset_ax_t2x = inset_axes(ax, width="100%", height="10%", loc="lower center", bbox_to_anchor=(0.0, -0.35, 1.0, 1.0), bbox_transform=ax.transAxes)
        toplotx = torch.tensor(toplotx)
        first_index = (toplotx < 1e-2).nonzero(as_tuple=True)[0][0]
        toplotxx = toplotx[:first_index]

        all_twodx = []
        for i in range(traj.shape[0]):
            curve = traj[i].squeeze()[:first_index]
            twodx = second_derivative(toplotxx, curve)
            all_twodx.append(twodx)
        twodx_agg = torch.stack(all_twodx).abs().mean(dim=0)
        #inset_ax_t2x.plot(toplotxx, twodx_agg, c="blue", linewidth=0.9)
        #inset_ax_t2x.axis('off')  # Disable axes
        #inset_ax_t2x.title.set_text("Curvature")
        #inset_ax_t2x.set_ylim(0, 5.0)
        #inset_ax_t2x.title.set_fontsize(11)

        #inset_ax_t2x.set_facecolor((1, 1, 1, 0))  # Make background transparent
        #inset_ax_t2x.set_position(ax.get_position())  # Align exactly with parent ax
        # -------------------

        ## GET rectified flow measure integral
        all_measure = []
        for i in range(traj.shape[0]):
            curve = traj[i].squeeze()[:first_index]
            dx = first_derivative(toplotxx, curve)
            all_measure.append(torch.square((curve[0] - curve[-1]) - dx))
        dx_agg = torch.stack(all_measure).mean(dim=0)
        integral = torch.trapz(reversed(dx_agg), reversed(toplotxx))
        
        # --- Combined inset with curvature and support + shared legend ---
        #inset_ax_combined = inset_axes(ax, width="100%", height="35%", loc="lower center", 
        #                                bbox_to_anchor=(0.0, -0.45, 1.0, 1.0), bbox_transform=ax.transAxes)

        # Primary y-axis: Curvature
        color_curv = "tab:green"
        curv_plot, = inset_ax_combined.plot(toplotxx, dx_agg, c=color_curv, linewidth=1.8, label="Curvature")
        inset_ax_combined.set_ylim(-0.8, 5)
        inset_ax_combined.axhline(0, color='black', linestyle='--', linewidth=0.8)
        inset_ax_combined.tick_params(axis='y', labelcolor=color_curv)
        #inset_ax_combined.xaxis.set_visible(False)

        # Secondary y-axis: Support
        ax_support = inset_ax_combined.twinx()
        color_support = "tab:red"
        base_support = sampler.reverse_process.forward_process.get_b(torch.tensor(1.0))
        support = sampler.reverse_process.forward_process.get_b(toplotxx)
        support = support / base_support
        support_plot, = ax_support.plot(toplotxx, support, c=color_support, linewidth=1.8, label="Support")
        ax_support.set_ylim(-0.2, 1.1)
        ax_support.tick_params(axis='y', labelcolor=color_support)

        # Optional: remove y-axis labels entirely if you just want color-coded curves
        inset_ax_combined.set_ylabel("")
        ax_support.set_ylabel("")

        # Add a legend to indicate which curve is which
        if "ISSNR" in testc:
            inset_ax_combined.legend(handles=[curv_plot, support_plot], loc="upper left",  
                                     fontsize=7, frameon=False)
            # move the legend a bit up
            inset_ax_combined.legend_.set_bbox_to_anchor((0., 1.1))
            inset_ax_combined.set_xticks([0.0, 1.0])
            inset_ax_combined.set_xticklabels(['0.0', '1.0'])
        else:
            # no ticks
            inset_ax_combined.set_xticks([])
            inset_ax_combined.set_xticklabels([])
        
        #write the integral as in the  above command but in green color
        inset_ax_combined.text(0.5, 0.8, f"Integral: {integral.item():.2f}",
                                 transform=inset_ax_combined.transAxes, 
                                 fontsize=8, ha='center', va='center', color=color_curv, 
                                 bbox=dict(facecolor='white', edgecolor='none', boxstyle='round,pad=0.3'))

        # Optional title
        #inset_ax_combined.set_title("Curvature & Support", fontsize=11)
        

        
        ticks = np.linspace(0, 1, traj.shape[0])

        if testc == "EDM":
            ticks = kve_schedule(ticks, sigma_min=0.0, sigma_max=1.0)
            # ax.set_xticks(ticks)
            # ax.set_xticklabels([(f"{x:.2f}" if (i == 0 or i == 20 or i == 18) else "") for (i, x) in enumerate(ticks)])

        # else:
        #     ax.set_xticks(ticks)
        #     ax.set_xticklabels([(f"{x:.2f}" if i % 10 == 0 else "") for (i, x) in enumerate(ticks)])
        ax.set_xticks(ticks)
        #ax.set_xticklabels([(f"{x:.2f}" if (x == 0 or x == 1.0) else "") for (i, x) in enumerate(ticks)])
        ax.set_xticklabels([])

        #if sigma_max > 1.0:
        # plot again
        #if testc == "EDM":
        inset_ax = inset_axes(ax, width="37%", height="50%", loc="upper left", bbox_to_anchor=(0.25, -0, 1.0, 1.0), bbox_transform=ax.transAxes)
        #else:
            #inset_ax = inset_axes(ax, width="37%", height="50%", loc="upper left", bbox_to_anchor=(0.12, -0.11, 1.0, 1.0), bbox_transform=ax.transAxes)

        inset_ax.yaxis.tick_left()
        inset_ax.xaxis.tick_top()

        topx0 = net.points_x0[-1].item()
        # if testc == "EDM" or testc == "SMLD" or testc == "EDM-UT":
        #     inset_ax.set_ylim(-3, 3)
        #     inset_ax.set_yticks([-2, 0, 2])
        #     ymin = -3
        #     ymax = 3
        # el
        # if testc == "VP-ISSNR[$\eta$=1 $\kappa$=2]" or testc == "VP-ISSNR[$\eta$=1.5 $\kappa$=1]" or testc == "VP-SMLD" or testc == "VP-EDM-UT":
        #     delta = 0.01
        #     inset_ax.set_ylim(topx0 - delta, topx0 + delta)
        #     inset_ax.set_yticks([np.round(topx0 - delta * 0.5, decimals=2), np.round(topx0 + delta * 0.5, decimals=2)])
        #     ymin = topx0 - delta
        #     ymax = topx0 + delta
        # else:
        deccc = 2

        if testc == "SMLD":
            delta = 0.01 # 0.03
            deccc = 3
        elif testc == "VP-SMLD":
            delta = 0.01 # 0.03
            deccc = 3
        elif testc == "OTFM":
            delta = 0.08
        elif testc == "VP-OTFM":
            delta = 0.08
        elif testc == "EDM-UT":
            delta = 0.02
        elif testc == "VP-EDM-UT":
            delta = 0.02
        elif testc == "EDM":
            delta = 1.05
        elif testc == "DDPM cos-v2.5":
            delta = 0.002
            deccc = 3
        elif testc == "DDPM cos-v1.0":
            delta = 0.3
        elif testc == "DDPM linear":
            delta = 0.5
        elif testc == "VP-ISSNR[$\eta$=1 $\kappa$=2]":
            delta = 0.03
        elif testc == "VP-ISSNR[$\eta$=1.5 $\kappa$=1]":
            delta = 0.015
        else:
            raise ValueError(f"Unknown testcase {testc}")

        # if testc == "EDM":
        #     delta = 1.05
        # elif testc == "DDPM cos-v1.0" or testc == "DDPM linear":
        #     delta = 0.5
        # elif "OTFM" in testc:
        #     delta = 0.3
        # elif testc == "DDPM cos-v2.5":
        #     delta = 0.003
        # else: #  or testc.startswith("VP-ISSNR") or testc.index("SMLD") > -1
        #     delta = 0.03

        inset_ax.set_ylim(topx0 - delta, topx0 + delta)
        paddd = -32 if (deccc == 3) else -25
        inset_ax.set_yticks([np.round(topx0 - delta * 0.5, decimals=deccc), np.round(topx0 + delta * 0.5, decimals=deccc)])
        ymin = topx0 - delta
        ymax = topx0 + delta

        inset_ax.yaxis.set_tick_params(direction='in', pad=paddd) 
        #inset_ax.set_xticks(ticks) # [0.0, 0.04]
        #inset_ax.set_xticklabels([(f"{x:.2f}" if (x == 0.0 or x >= 0.03) else "") for (i, x) in enumerate(ticks)])
        inset_ax.set_xticks([]) # [0.0, 0.04]
        inset_ax.set_xticklabels([])

        inset_ax.set_xlim(-0.0065, 0.05)

        tt = np.logspace(-4, np.log10(0.05), 45)
        xx = np.linspace(ymin, ymax, 55)
        TT, XX = np.meshgrid(tt, xx)

        Z = net.log_prob(torch.from_numpy(XX.reshape(-1)), torch.from_numpy(TT.reshape(-1)))
        Z = Z.reshape((TT.shape[0], TT.shape[1])).numpy()

        if contours:
            inset_ax.contourf(TT, XX, Z, levels=1000, vmin=vmin / 2, cmap=custom_oranges)

        for i in range(traj.shape[0]):
            toplot = traj[i].squeeze().detach().numpy()
            #toplotx = torch.linspace(1, 0, toplot.shape[0])

            if i == traj.shape[0] - 1:
                pass
                # uncomment to plot marker points
                # inset_ax.plot(toplotx, toplot, c="black", linewidth=0.3, marker='o', markersize=2)
            else:
                inset_ax.plot(toplotx, toplot, c="black", linewidth=0.3)

        # end inset

        ax.set_ylim(-ymaxx, ymaxx)
        title = f"{testc}"
        if "VP-ISSNR" in testc:
            title = "VP-ISSNR"
        if testc[0] == "V":
            title = "Our " + title
        ax.set_title(title)  # - {ndp} deltas
        # size of title 
        ax.title.set_size(10)
        # bold title
        ax.title.set_weight('bold')
    
    plt.savefig(f"figures/fig1_toy_example{iiii}_{ndp}_deltas.pdf", bbox_inches='tight', pad_inches=0)
    plt.savefig(f"figures/fig1_toy_example{iiii}_{ndp}_deltas.png", bbox_inches='tight', pad_inches=0)
    plt.tight_layout()
    plt.show()

In [ ]:
import torch
import matplotlib.pyplot as plt

def kve_schedule(
    i: torch.Tensor, sigma_min: float = 0.002, sigma_max: float = 30.0, rho: float = 7.0
) -> torch.Tensor:
    """
    Variance Exploding schedule from Karras et al. 2022
    https://arxiv.org/abs/2206.00364.

    Args:
        i: index of time step in [0.,1.].
        sigma_min: minimum std.
        sigma_max: maximum std.
        rho: steepness of exponential variance.
    """
    return (
        sigma_max ** (1 / rho)
        + (1.0 - i) * (sigma_min ** (1 / rho) - sigma_max ** (1 / rho))
    ) ** rho

i = torch.linspace(0, 1, 100)
for rho in range(1, 8):
    # get time step as function of t
    mapping = kve_schedule(i, sigma_min=0.0, sigma_max=1.0, rho=rho)
    # dmap = torch.diff(mapping)
    plt.plot(i, mapping, label=r"$\rho$ = " + str(rho), linestyle="-" if rho==7 else "--", linewidth=2.5 if rho==7 else 1)
    plt.legend()
    #plt.plot(i, )
    plt.ylabel("Time (t)")
    plt.xlabel("Index (i)")

plt.savefig("figures/kve_schedule.pdf", bbox_inches='tight', pad_inches=0)